# Tính tâm x,y từ ảnh mask

In [48]:
import cv2
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

# =============================================================================
# CÁC THAM SỐ CẦN CẤU HÌNH
# =============================================================================

# 1. Đường dẫn đến thư mục chứa các file mask .npy
MASK_DIR = "/Users/angelinacu/Desktop/Study/Viettel/ThiSinh/Train/masks_npy"

# 2. Đường dẫn file CSV đầu ra
OUTPUT_CSV = "/Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/coco_label/centroids_pack_final.csv"

# =============================================================================
# HÀM CHÍNH (MAIN SCRIPT) - ĐÃ CẬP NHẬT LOGIC LOẠI BỎ
# =============================================================================

def calculate_centroids_from_npy_masks():
    results_list = []
    print(f"Bắt đầu quét thư mục mask: {MASK_DIR}")
    
    try:
        mask_files = [f for f in os.listdir(MASK_DIR) if f.endswith('.npy')]
    except FileNotFoundError:
        print(f"LỖI: Không tìm thấy thư mục: {MASK_DIR}")
        return
    
    if not mask_files:
        print(f"LỖI: Không tìm thấy file '.npy' nào.")
        return

    print(f"Tìm thấy {len(mask_files)} file mask. Bắt đầu tính toán...")

    skipped_masks = 0

    for filename in tqdm(mask_files, desc="Đang xử lý"):
        try:
            # --- 1. Tách tên file ---
            image_base_name = filename.split('_')[0]
            output_image_name = f"{image_base_name}.png"
            
            # --- 2. Tải file mask ---
            mask_path = os.path.join(MASK_DIR, filename)
            mask = np.load(mask_path)
            
            # Chuyển sang uint8 (0 và 255)
            mask_uint8 = (mask * 255).astype(np.uint8)

            # --- 3. Tính toán tâm hình học (Geometric Centroid) ---
            M = cv2.moments(mask_uint8)
            if M["m00"] > 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                
                # --- [MỚI] 4. Kiểm tra tâm có nằm trong mask không ---
                # Đảm bảo toạ độ không vượt quá kích thước ảnh
                h, w = mask_uint8.shape
                cx = min(max(cx, 0), w - 1)
                cy = min(max(cy, 0), h - 1)

                # Nếu pixel tại (cy, cx) là nền (giá trị 0) -> LOẠI BỎ
                if mask_uint8[cy, cx] == 0:
                    skipped_masks += 1
                    continue # Bỏ qua mask này, sang mask tiếp theo

                # --- 5. Lưu kết quả (chỉ khi tâm hợp lệ) ---
                results_list.append({
                    'image_name': output_image_name,
                    '2d_x': cx,
                    '2d_y': cy
                })
                
        except Exception as e:
            print(f"Lỗi file {filename}: {e}")

    # --- 6. Lưu file CSV ---
    if not results_list:
        print("Không tính được tâm nào hợp lệ.")
        return

    df_results = pd.DataFrame(results_list)
    df_results = df_results[['image_name', '2d_x', '2d_y']]
    df_results = df_results.sort_values(by=['image_name', '2d_x'])
    
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    df_results.to_csv(OUTPUT_CSV, index=False)
    
    print(f"\nHoàn tất! Đã lưu {len(df_results)} tâm hợp lệ vào CSV.")
    print(f"Đã loại bỏ {skipped_masks} mask do tâm nằm ngoài vật thể.")

if __name__ == "__main__":
    calculate_centroids_from_npy_masks()

Bắt đầu quét thư mục mask: /Users/angelinacu/Desktop/Study/Viettel/ThiSinh/Train/masks_npy
Tìm thấy 1248 file mask. Bắt đầu tính toán...


Đang xử lý: 100%|██████████| 1248/1248 [00:01<00:00, 759.00it/s]



Hoàn tất! Đã lưu 1232 tâm hợp lệ vào CSV.
Đã loại bỏ 16 mask do tâm nằm ngoài vật thể.


# Lọc các vật trong ROI

In [49]:
import pandas as pd
import os

# =================================================================
# Cấu hình đường dẫn và ROI
# =================================================================

CSV_FILE_PATH = '/Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/coco_label/centroids_pack_final.csv'

# Tọa độ ROI mới
ROI_X = 550
ROI_Y = 150
ROI_WIDTH = 300
ROI_HEIGHT = 330

# =================================================================
# Chức năng chính
# =================================================================

def filter_centroids_by_roi(file_path, roi_x, roi_y, roi_w, roi_h):
    """
    Đọc file CSV, lọc các tọa độ Centroid nằm ngoài ROI, và cập nhật file CSV.
    """
    if not os.path.exists(file_path):
        print(f"LỖI: Không tìm thấy file CSV tại đường dẫn: {file_path}")
        return

    try:
        # 1. Đọc file CSV
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"LỖI khi đọc file CSV: {e}")
        return

    # Tính toán ranh giới phải và dưới của ROI
    roi_right = roi_x + roi_w
    roi_bottom = roi_y + roi_h
    
    print(f"Bắt đầu lọc dữ liệu. ROI Boundaries: X=[{roi_x}, {roi_right}), Y=[{roi_y}, {roi_bottom})")
    print(f"Tổng số hàng ban đầu: {len(df)}")

    # 2. Định nghĩa điều kiện lọc (chỉ giữ lại các Centroid nằm trong ROI)
    # Centroid (Cx, Cy) nằm trong ROI nếu:
    # Cx >= ROI_X AND Cx < ROI_RIGHT AND Cy >= ROI_Y AND Cy < ROI_BOTTOM
    
    condition = (df['2d_x'] >= roi_x) & \
                (df['2d_x'] < roi_right) & \
                (df['2d_y'] >= roi_y) & \
                (df['2d_y'] < roi_bottom)

    # 3. Áp dụng bộ lọc
    df_filtered = df[condition]
    
    # 4. Kiểm tra và cập nhật file
    rows_removed = len(df) - len(df_filtered)
    
    if rows_removed > 0:
        # Ghi đè file CSV với dữ liệu đã lọc
        df_filtered.to_csv(file_path, index=False)
        print(f"✅ Hoàn thành lọc! Đã xoá {rows_removed} tọa độ Centroid nằm ngoài ROI.")
        print(f"Tổng số hàng sau khi lọc: {len(df_filtered)}")
    else:
        print("ℹ️ Không có tọa độ nào nằm ngoài ROI. File CSV không thay đổi.")

if __name__ == '__main__':
    # Đảm bảo cài đặt thư viện pandas: pip install pandas
    filter_centroids_by_roi(CSV_FILE_PATH, ROI_X, ROI_Y, ROI_WIDTH, ROI_HEIGHT)

Bắt đầu lọc dữ liệu. ROI Boundaries: X=[550, 850), Y=[150, 480)
Tổng số hàng ban đầu: 1232
✅ Hoàn thành lọc! Đã xoá 538 tọa độ Centroid nằm ngoài ROI.
Tổng số hàng sau khi lọc: 694


# Chuyển đổi tâm 2D sang 3D

In [64]:
import cv2
import numpy as np
import pandas as pd
import os
from sklearn.linear_model import RANSACRegressor
from tqdm import tqdm
from scipy.ndimage import median_filter # Thêm thư viện cho median filter

# =============================================================================
# CÁC THAM SỐ CẦN CẤU HÌNH
# =============================================================================

DEPTH_IMAGE_DIR = "/Users/angelinacu/Desktop/Study/Viettel/ThiSinh/Train/depth"
INPUT_CSV = "/Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/coco_label/centroids_pack_final.csv"
OUTPUT_CSV = "/Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/Submission_3D.csv"
DEPTH_SCALE = 1000.0
Z_THRESHOLD_METERS = 0.005 

color_intrinsics = {
    'width': 1280, 'height': 720,
    'fx': 643.90087890625, 'fy': 643.1365356445312,
    'cx': 650.2113037109375, 'cy': 355.79559326171875,
    'model': "distortion.inverse_brown_conrady",
    'coeffs': [-0.05658450722694397, 0.06544225662946701, -0.0008694113348610699, 0.00016751799557823688, -0.020957745611667633]
}

ROBOT_COORD_3D = np.array([0.6524, -0.2263, 0.9340])
REFINEMENT_RADIUS = 0.1
MIN_NEIGHBORS_FOR_RANSAC = 2000
RANSAC_RESIDUAL_THRESHOLD = 0.005

# =============================================================================
# HÀM HỖ TRỢ 
# =============================================================================

def deproject_with_inverse_model(u_distorted, v_distorted, depth_meters, intrinsics):
    # (Giữ nguyên hàm này)
    fx = intrinsics['fx']; fy = intrinsics['fy']; cx = intrinsics['cx']; cy = intrinsics['cy']
    k1, k2, p1, p2, k5 = intrinsics['coeffs']
    x_distorted_normalized = (u_distorted - cx) / fx
    y_distorted_normalized = (v_distorted - cy) / fy
    r2 = x_distorted_normalized**2 + y_distorted_normalized**2; r4 = r2**2; r6 = r2**3
    radial_distortion = 1 + k1*r2 + k2*r4 + k5*r6
    tangential_distortion_x = 2*p1*x_distorted_normalized*y_distorted_normalized + p2*(r2 + 2*x_distorted_normalized**2)
    tangential_distortion_y = p1*(r2 + 2*y_distorted_normalized**2) + 2*p2*x_distorted_normalized*y_distorted_normalized
    x_undistorted_normalized = x_distorted_normalized * radial_distortion + tangential_distortion_x
    y_undistorted_normalized = y_distorted_normalized * radial_distortion + tangential_distortion_y
    z_c = depth_meters; x_c = x_undistorted_normalized * z_c; y_c = y_undistorted_normalized * z_c
    return x_c, y_c, z_c

# --- [MỚI] Hàm Pre-processing Depth (Fill Holes) ---
def preprocess_depth_image(depth_image_raw, scale=1000.0, fill_holes=True):
    """
    Chuyển đổi ảnh depth sang mét và lấp đầy các lỗ trống (holes).
    """
    # Chuyển sang float32 và đơn vị mét
    Z = depth_image_raw.astype(np.float32) / scale
    
    # Tạo mask cho các pixel hợp lệ ban đầu
    valid = (Z > 0.01) & (Z < 5.0) & np.isfinite(Z)
    
    if fill_holes:
        # Áp dụng bộ lọc trung vị để lấp lỗ
        Z_filled = median_filter(Z, size=3)
        
        fill_mask = (~valid) & (cv2.dilate(valid.astype(np.uint8), np.ones((3,3), np.uint8)) > 0)
        
        # Gán giá trị đã lấp vào ảnh gốc
        Z[fill_mask] = Z_filled[fill_mask]
    
    # Trả về ảnh độ sâu (tính bằng mét) đã được xử lý
    return Z

def build_full_point_cloud_vectorized(depth_meters, intrinsics):
    """
    [CẬP NHẬT] Nhận đầu vào là ảnh depth ĐÃ ĐƯỢC PRE-PROCESS (đơn vị mét).
    """
    height, width = depth_meters.shape
    v_coords, u_coords = np.indices((height, width))
    
    # Lấy mask các pixel hợp lệ (độ sâu > 0 sau khi đã fill holes)
    valid_mask = depth_meters > 0
    
    u_distorted = u_coords[valid_mask]
    v_distorted = v_coords[valid_mask]
    z_c = depth_meters[valid_mask] # Đã là float mét
    
    if z_c.size == 0: return np.array([])

    fx = intrinsics['fx']; fy = intrinsics['fy']; cx = intrinsics['cx']; cy = intrinsics['cy']
    k1, k2, p1, p2, k5 = intrinsics['coeffs']
    
    # Vectorized deprojection (giữ nguyên logic cũ)
    x_dist_n = (u_distorted - cx) / fx; y_dist_n = (v_distorted - cy) / fy
    r2 = x_dist_n**2 + y_dist_n**2; r4 = r2**2; r6 = r2**3
    rad_dist = 1 + k1*r2 + k2*r4 + k5*r6
    tan_dist_x = 2*p1*x_dist_n*y_dist_n + p2*(r2 + 2*x_dist_n**2)
    tan_dist_y = p1*(r2 + 2*y_dist_n**2) + 2*p2*x_dist_n*y_dist_n
    x_undist_n = x_dist_n * rad_dist + tan_dist_x
    y_undist_n = y_dist_n * rad_dist + tan_dist_y
    x_c = x_undist_n * z_c; y_c = y_undist_n * z_c
    
    return np.stack([x_c, y_c, z_c], axis=-1)

def refine_z_with_ransac(full_point_cloud, pred_x, pred_y, pred_z):
    # (Giữ nguyên hàm này)
    if full_point_cloud.size == 0: return pred_z
    distances = np.linalg.norm(full_point_cloud - [pred_x, pred_y, pred_z], axis=1)
    neighborhood_points = full_point_cloud[distances <= REFINEMENT_RADIUS]
    if len(neighborhood_points) < MIN_NEIGHBORS_FOR_RANSAC: return pred_z
    try:
        ransac = RANSACRegressor(residual_threshold=RANSAC_RESIDUAL_THRESHOLD, random_state=42)
        ransac.fit(neighborhood_points[:, [0, 1]], neighborhood_points[:, 2])
        z_refined = ransac.predict(np.array([[pred_x, pred_y]]))[0]
        if abs(z_refined - pred_z) > 0.05: return pred_z
        return z_refined
    except: return pred_z

# =============================================================================
# HÀM CHÍNH
# =============================================================================

def process_coordinates():
    print(f"Bắt đầu xử lý tệp: {INPUT_CSV}")
    try: df = pd.read_csv(INPUT_CSV)
    except: return

    all_3d_points = []
    # Cache lưu trữ ảnh depth ĐÃ PRE-PROCESS (tính bằng mét)
    depth_meters_cache = {} 

    # --- GIAI ĐOẠN 1: Chiếu 2D -> 3D (Sử dụng ảnh depth đã fill holes) ---
    print("Đang tải ảnh depth, fill holes và chiếu 2D->3D...")
    for index, row in tqdm(df.iterrows(), total=len(df)):
        try:
            image_name = row['image_name']
            u_d = float(row['2d_x']); v_d = float(row['2d_y'])
            
            if image_name not in depth_meters_cache:
                raw_img = cv2.imread(os.path.join(DEPTH_IMAGE_DIR, image_name), cv2.IMREAD_UNCHANGED)
                if raw_img is None: continue
                # [QUAN TRỌNG] Áp dụng pre-processing ngay khi tải ảnh
                depth_meters_cache[image_name] = preprocess_depth_image(raw_img, DEPTH_SCALE, fill_holes=True)
            
            depth_meters_img = depth_meters_cache[image_name]
            u_i, v_i = int(round(u_d)), int(round(v_d))
            if not (0 <= v_i < depth_meters_img.shape[0] and 0 <= u_i < depth_meters_img.shape[1]): continue
            
            # Lấy giá trị Z từ ảnh ĐÃ FILL HOLES
            z_val = depth_meters_img[v_i, u_i]
            if z_val <= 0: continue # Vẫn kiểm tra nếu fill holes không lấp được
            
            x, y, z = deproject_with_inverse_model(u_d, v_d, z_val, color_intrinsics)
            all_3d_points.append({'image_filename': image_name, 'x': x, 'y': y, 'z': z, 'z_original': z})
        except: continue

    df_all = pd.DataFrame(all_3d_points)
    if df_all.empty: return

    # --- GIAI ĐOẠN 2: Tinh chỉnh Z (RANSAC) ---
    print("Đang tinh chỉnh Z bằng RANSAC...")
    refined_z_list = []
    for img_name, group in tqdm(df_all.groupby('image_filename')):
        # Dùng ảnh depth ĐÃ FILL HOLES từ cache để xây dựng point cloud
        full_pc = build_full_point_cloud_vectorized(depth_meters_cache[img_name], color_intrinsics)
        for idx, row in group.iterrows():
            z_ref = refine_z_with_ransac(full_pc, row['x'], row['y'], row['z_original'])
            refined_z_list.append({'index': idx, 'z_refined': z_ref})
            
    df_all = df_all.join(pd.DataFrame(refined_z_list).set_index('index'))
    df_all['z'] = df_all['z_refined']

    # --- GIAI ĐOẠN 3: Lọc (Filter) ---
    print("Đang lọc kết quả cuối cùng...")
    df_all['min_z'] = df_all.groupby('image_filename')['z'].transform('min')
    
    cands = df_all[df_all['z'] <= (df_all['min_z'] + Z_THRESHOLD_METERS)].copy()
    
    cands['dist'] = np.linalg.norm(cands[['x','y','z']].values - ROBOT_COORD_3D, axis=1)
    
    final_df = cands.sort_values(['dist'], ascending= False).drop_duplicates('image_filename')

    final_df['image_filename'] = 'image_' + final_df['image_filename'].str.replace('image_', '', regex=False)
    final_df[['image_filename', 'x', 'y', 'z']].to_csv(OUTPUT_CSV, index=False, float_format='%.3f')
    print(f"Hoàn tất! Đã lưu vào {OUTPUT_CSV}")

if __name__ == "__main__":
    process_coordinates()

Bắt đầu xử lý tệp: /Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/coco_label/centroids_pack_final.csv
Đang tải ảnh depth, fill holes và chiếu 2D->3D...


  0%|          | 0/694 [00:00<?, ?it/s]

100%|██████████| 694/694 [00:23<00:00, 30.07it/s]


Đang tinh chỉnh Z bằng RANSAC...


100%|██████████| 369/369 [01:04<00:00,  5.74it/s]

Đang lọc kết quả cuối cùng...
Hoàn tất! Đã lưu vào /Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/Submission_3D.csv


# Tính Normal Vector


In [47]:
import pandas as pd
import open3d as o3d
import numpy as np
import os
import re  # Để trích xuất tên file
from tqdm import tqdm  # Để xem thanh tiến trình

# --- HẰNG SỐ CẤU HÌNH ---
# [SỬA] Cả đầu vào và đầu ra đều là cùng một file
CSV_PATH = "/Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/Submission_3D.csv"
PLY_DIR = "/Users/angelinacu/Desktop/Study/Viettel/ThiSinh/Test/ply"
OUTPUT_CSV_PATH = "/Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/Submission_3D.csv"

# Vector pháp tuyến mặc định nếu tính toán thất bại
DEFAULT_NORMAL = np.array([0.0, 0.0, 1.0])

# Ma trận chuyển đổi 4x4 cho open3d
TRANSFORM_MATRIX = np.array([
    [1, 0, 0, 0],
    [0, -1, 0, 0],
    [0, 0, -1, 0],
    [0, 0, 0, 1]
])

# --- THAM SỐ THUẬT TOÁN (CỐ ĐỊNH) ---
K_NEIGHBORS = 4000
RANSAC_DISTANCE_THRESHOLD = 0.008
RANSAC_N = 3
RANSAC_ITERATIONS = 1000
Z_FILTER_TOLERANCE = 0.005 
SURFACE_VARIATION_THRESHOLD = 0.002


def get_ply_path_from_image(image_filename, base_ply_dir):
    """
    Chuyển đổi tên file từ 'image_XXXX.png' thành đường dẫn '.../XXXX.ply'.
    """
    match = re.search(r'image_(\d+)\.png', image_filename)
    if not match:
        return None
    ply_name = f"{match.group(1)}.ply"
    return os.path.join(base_ply_dir, ply_name)

def calculate_normal_and_curvature(pcd, center_point, k, dist_thresh, n_pts, iters, 
                                   z_tolerance):
    """
    NÂNG CẤP: Lọc K lân cận theo chênh lệch trục Z (nếu z_tolerance > 0).
    1. Tìm k lân cận (KNN).
    2. [MỚI] Lọc lân cận theo ngưỡng Z.
    3. Chạy RANSAC để tìm 'inliers' từ tập đã lọc.
    4. Chạy PCA trên 'inliers'.
    5. Trả về pháp tuyến VÀ độ cong.
    """
    try:
        pcd_tree = o3d.geometry.KDTreeFlann(pcd)
        [k_found, idx, _] = pcd_tree.search_knn_vector_3d(center_point, k)
        
        if k_found < n_pts:
            return None, None

        neighbor_points = np.asarray(pcd.points)[idx, :]
        

        if z_tolerance > 0:
            center_z = center_point[2]
            neighbor_z_values = neighbor_points[:, 2] # Lấy tất cả tọa độ Z của lân cận
            
            # Tạo một mask (mặt nạ) boolean
            z_mask = np.abs(neighbor_z_values - center_z) < z_tolerance
            
            # Lọc các điểm
            filtered_neighbor_points = neighbor_points[z_mask]
        else:
            # Nếu z_tolerance = 0, dùng tất cả lân cận
            filtered_neighbor_points = neighbor_points

        # Cần ít nhất n_pts điểm SAU KHI LỌC
        if filtered_neighbor_points.shape[0] < n_pts:
            return None, None
        # --- [KẾT THÚC BỔ SUNG] ---

        pcd_neighbors = o3d.geometry.PointCloud()
        # Sử dụng các điểm ĐÃ LỌC
        pcd_neighbors.points = o3d.utility.Vector3dVector(filtered_neighbor_points)

        plane_model, inlier_indices = pcd_neighbors.segment_plane(
            distance_threshold=dist_thresh,
            ransac_n=n_pts,
            num_iterations=iters
        )
        
        # Lấy inliers từ TẬP ĐÃ LỌC
        inlier_points = filtered_neighbor_points[inlier_indices, :]
        
        # Cần ít nhất n_pts (3) điểm để tính PCA ổn định
        if inlier_points.shape[0] < n_pts:
            return None, None

        # --- TÍNH TOÁN PCA VÀ ĐỘ CONG (giữ nguyên) ---
        covariance_matrix = np.cov(inlier_points, rowvar=False)
        eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
        
        lambda_0 = eigenvalues[0]
        lambda_1 = eigenvalues[1]
        lambda_2 = eigenvalues[2]
        sum_eigenvalues = lambda_0 + lambda_1 + lambda_2
        
        if sum_eigenvalues == 0:
            return None, None 

        surface_variation = lambda_0 / sum_eigenvalues
        normal_vector = eigenvectors[:, 0] 
        
        return normal_vector, surface_variation

    except Exception as e:
        # print(f"Lỗi: {e}")
        return None, None


def main():
    # 1. Đọc file CSV (chỉ chứa x, y, z)
    try:
        df = pd.read_csv(CSV_PATH)
    except FileNotFoundError:
        print(f"LỖI: Không tìm thấy file CSV tại: {CSV_PATH}")
        return
    except KeyError:
        print(f"LỖI: File {CSV_PATH} dường như đã có cột Rx,Ry,Rz? Xóa file đó đi và chạy lại script 2D-to-3D trước.")
        return


    # Danh sách lưu kết quả cho file Submission (x,y,z,Rx,Ry,Rz)
    submission_list = []



    print(f"Bắt đầu tính toán Normal Vector (RANSAC+PCA) cho {len(df)} ảnh...")
    
    # 2. Lặp qua từng hàng trong file CSV (từng mẫu)
    for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Đang xử lý file"):
        image_name = row['image_filename']
        center_point = np.array([row['x'], row['y'], row['z']])
        
        
        # Đặt giá trị Rx, Ry, Rz mặc định
        final_normal = DEFAULT_NORMAL
        
        ply_path = get_ply_path_from_image(image_name, PLY_DIR)
        
        pcd = None
        if ply_path and os.path.exists(ply_path):
            try:
                pcd = o3d.io.read_point_cloud(ply_path)
                if pcd.is_empty():
                    pcd = None
                else:
                    pcd.transform(TRANSFORM_MATRIX)
            except Exception as e:
                pcd = None 
                

        current_z_tolerance = Z_FILTER_TOLERANCE

        # Chỉ tính toán nếu PCL được tải thành công
        if pcd is not None:
            calculated_normal, surface_variation = calculate_normal_and_curvature(
                pcd, 
                center_point, 
                K_NEIGHBORS,
                RANSAC_DISTANCE_THRESHOLD,
                RANSAC_N, 
                RANSAC_ITERATIONS,
                current_z_tolerance
            )
            
            # Nếu tính toán thành công, cập nhật giá trị
            if calculated_normal is not None:
                final_normal = calculated_normal # Dùng vector đã tính
        
        surface_variation = None
        if surface_variation is not None and surface_variation <= SURFACE_VARIATION_THRESHOLD:
            final_normal = DEFAULT_NORMAL
        
        # Luôn luôn thêm vào submission_list (dùng giá trị mặc định hoặc đã tính)
        submission_list.append({
            'image_filename': image_name,
            'x': center_point[0],
            'y': center_point[1],
            'z': center_point[2],
            'Rx': final_normal[0],
            'Ry': final_normal[1],
            'Rz': final_normal[2]
        })

    # --- KẾT THÚC VÒNG LẶP CHÍNH ---

    # 5. Tạo DataFrame và xuất file Submission
    if not submission_list:
        print("\nKhông xử lý thành công bất kỳ file nào.")
        return

    # A. Lưu file CSV submission
    results_df = pd.DataFrame(submission_list)
    
    # Sắp xếp lại cột để đảm bảo đúng thứ tự
    columns_ordered = ['image_filename', 'x', 'y', 'z', 'Rx', 'Ry', 'Rz']
    results_df = results_df[columns_ordered]
    
    # Ghi đè lên file OUTPUT_CSV_PATH (cũng là file CSV_PATH)
    results_df.to_csv(OUTPUT_CSV_PATH, index=False, float_format='%.3f')
    print(f"\nĐã tính toán và ghi đè 7 cột (đã làm tròn 3 số) vào file: {OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    main()

Bắt đầu tính toán Normal Vector (RANSAC+PCA) cho 20 ảnh...


Đang xử lý file: 100%|██████████| 20/20 [00:07<00:00,  2.59it/s]


Đã tính toán và ghi đè 7 cột (đã làm tròn 3 số) vào file: /Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/Submission_3D.csv


# Compare

In [65]:
import pandas as pd
import numpy as np
import os

# --- 1. Định nghĩa Tên tệp và Hằng số ---
PREDICT_CSV = "/Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/Submission_3D.csv"
GT_CSV = "/Users/angelinacu/Desktop/Study/Viettel/ThiSinh/Train/Public train.csv"

# Tên file output mới để lưu kết quả so sánh
OUTPUT_CSV_PATH = "/Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/comparison_metrics_normalized.csv"

# [THÊM MỚI] Ngưỡng sai số hợp lệ (0.05 mét = 5cm)
VALIDITY_THRESHOLD = 0.05

# --- 2. Hàm tính toán ---
def calculate_metrics():
    """
    Tải file dự đoán và file ground truth, so sánh,
    tính MCE chuẩn hóa, lưu kết quả và in ra các chỉ số trung bình.
    """
    
    # --- 3. Tải dữ liệu ---
    try:
        cols_to_use = ['image_filename', 'x', 'y', 'z']
        df_predict = pd.read_csv(PREDICT_CSV, usecols=cols_to_use)
        df_gt = pd.read_csv(GT_CSV, usecols=cols_to_use)
        
    except FileNotFoundError as e:
        print(f"LỖI: Không tìm thấy tệp: {e.filename}")
        return
    except ValueError as e:
        print(f"LỖI: Tệp CSV có thể thiếu cột. {e}")
        print(f"Hãy đảm bảo cả hai tệp đều có cột: {cols_to_use}")
        return
    except Exception as e:
        print(f"LỖI khi đọc CSV: {e}")
        return

    print(f"Đã tải {len(df_predict)} dự đoán từ: {os.path.basename(PREDICT_CSV)}")
    print(f"Đã tải {len(df_gt)} ground truth từ: {os.path.basename(GT_CSV)}")

    # --- 4. Gộp (Merge) hai DataFrame ---
    df_merged = pd.merge(
        df_predict, 
        df_gt, 
        on='image_filename', 
        suffixes=('_pred', '_gt')
    )
    
    if df_merged.empty:
        print("LỖI: Không có image_filename nào trùng khớp giữa hai tệp.")
        return

    print(f"Tìm thấy {len(df_merged)} ảnh trùng khớp để so sánh.")

    # --- 5. Tính toán Sai số (Error) ---
    
    # 5a. Tính sai số (deviation) cho từng trục (giữ nguyên)
    df_merged['x_error'] = df_merged['x_pred'] - df_merged['x_gt']
    df_merged['y_error'] = df_merged['y_pred'] - df_merged['y_gt']
    df_merged['z_error'] = df_merged['z_pred'] - df_merged['z_gt']

    # 5b. Tính khoảng cách Euclidean 3D thô (để làm cơ sở)
    pred_coords = df_merged[['x_pred', 'y_pred', 'z_pred']].to_numpy()
    gt_coords = df_merged[['x_gt', 'y_gt', 'z_gt']].to_numpy()
    df_merged['mce_distance'] = np.linalg.norm(pred_coords - gt_coords, axis=1)

    # 5c. [LOGIC MỚI] Tính MCE chuẩn hóa và giới hạn
    # Chia sai số thô cho ngưỡng, sau đó giới hạn giá trị tối đa là 1.0
    df_merged['normalized_mce'] = (df_merged['mce_distance'] / VALIDITY_THRESHOLD).clip(upper=1.0)


    # --- 6. Lưu file CSV kết quả chi tiết ---
    
    # [SỬA] Thêm cột mới vào file output
    output_columns = [
        'image_filename', 
        'x_pred', 'x_gt', 'x_error',
        'y_pred', 'y_gt', 'y_error',
        'z_pred', 'z_gt', 'z_error',
        'mce_distance',     # Sai số thô (mét)
        'normalized_mce'    # Sai số đã chuẩn hóa và giới hạn
    ]
    
    df_results = df_merged[output_columns]
    
    try:
        df_results.to_csv(OUTPUT_CSV_PATH, index=False, float_format='%.3f')
        print(f"\nĐã lưu kết quả so sánh chi tiết vào: {OUTPUT_CSV_PATH}")
    except Exception as e:
        print(f"\nLỖI: Không thể lưu file CSV: {e}")

    # --- 7. Tính toán và In ra các chỉ số trung bình đã cập nhật ---
    
    # [SỬA] Chỉ số chính là MCE chuẩn hóa trung bình
    mean_normalized_mce = df_merged['normalized_mce'].mean()
    
    # [THÊM MỚI] Tính tỷ lệ hợp lệ
    accuracy_rate = (df_merged['mce_distance'] < VALIDITY_THRESHOLD).mean() * 100
    
    # Giữ lại các chỉ số thô để tham khảo
    mean_raw_mce = df_merged['mce_distance'].mean()
    mean_x_deviation = df_merged['x_error'].mean()
    mean_y_deviation = df_merged['y_error'].mean()
    
    print("\n" + "="*50)
    print("--- 🏆 CHỈ SỐ CHUẨN HÓA (Normalized MCE) ---")
    print(f"Điểm MCE chuẩn hóa trung bình: {mean_normalized_mce:.3f} (0 = hoàn hảo, 1 = sai số lớn)")
    print(f"Tỷ lệ hợp lệ (sai số < {VALIDITY_THRESHOLD*1000:.0f}mm): {accuracy_rate:.2f}%")
    print("="*50)
    
    print("\n--- 📊 CHỈ SỐ THÔ (Để tham khảo) ---")
    print(f"MCE trung bình (Khoảng cách 3D thô): {mean_raw_mce:.3f} mét")
    print(f"Độ lệch X trung bình (pred - gt): {mean_x_deviation:.3f} mét")
    print(f"Độ lệch Y trung bình (pred - gt): {mean_y_deviation:.3f} mét")


# --- Chạy hàm chính ---
if __name__ == "__main__":
    calculate_metrics()

Đã tải 369 dự đoán từ: Submission_3D.csv
Đã tải 350 ground truth từ: Public train.csv
Tìm thấy 349 ảnh trùng khớp để so sánh.

Đã lưu kết quả so sánh chi tiết vào: /Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/comparison_metrics_normalized.csv

--- 🏆 CHỈ SỐ CHUẨN HÓA (Normalized MCE) ---
Điểm MCE chuẩn hóa trung bình: 0.214 (0 = hoàn hảo, 1 = sai số lớn)
Tỷ lệ hợp lệ (sai số < 50mm): 96.28%

--- 📊 CHỈ SỐ THÔ (Để tham khảo) ---
MCE trung bình (Khoảng cách 3D thô): 0.016 mét
Độ lệch X trung bình (pred - gt): -0.003 mét
Độ lệch Y trung bình (pred - gt): -0.000 mét
